# 1 Tagesschau Golstandard Evaluation (Nachrichten vom 17.09.26)
Nachrichten wurden mit RSS extrahiert.

## 1.1 Evaluation

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "nameless-0/GER_frame-detect_mgbert134M"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

config.json:   0%|          | 0.00/2.69k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/729k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  539MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded on:", device)
print("Number of labels:", model.config.num_labels)

Model loaded on: cuda
Number of labels: 15


In [16]:
df_gold = pd.read_excel("goldstandard.xlsx")
df_predict = df_gold.rename(columns={0:'Headline'})['Headline']
df_predict

,Headline
0,Warum die SPD jetzt auf Manuela Schwesig hofft
1,Livestream: Die Nachrichten auf tagesschau24
2,Hohe Spritpreise: So will die Bundesregierung ...
3,Jemen: Warum die Huthi-Miliz den Welthandel ge...
4,Kosovos Ex-Präsident Thaçi zu 25 Jahren Haft v...
...,...
65,Marktbericht: DAX rückt nach US-Zinsentscheid ...
66,"Wetter Deutschland: Wolken, Temperaturen, Wind..."
67,Regenradar Deutschland: Aktueller Niederschlag
68,11KM-Podcast: AfD in Sachsen-Anhalt - über Mac...


In [17]:
df_gold

,Unnamed: 0,0,Economic,Capacity and resources,Morality,Fairness and equality,"Legality, constitutionality and jurisprudence",Policy prescription and evaluation,Crime and punishment,Security and defense,Health and safety,Quality of life,Cultural identity,Public opinion,Political,External regulation and reputation,other
0,0,Warum die SPD jetzt auf Manuela Schwesig hofft,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,1,Livestream: Die Nachrichten auf tagesschau24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,2,Hohe Spritpreise: So will die Bundesregierung ...,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,3,Jemen: Warum die Huthi-Miliz den Welthandel ge...,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
4,4,Kosovos Ex-Präsident Thaçi zu 25 Jahren Haft v...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,65,Marktbericht: DAX rückt nach US-Zinsentscheid ...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
66,66,"Wetter Deutschland: Wolken, Temperaturen, Wind...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
67,67,Regenradar Deutschland: Aktueller Niederschlag,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
68,68,11KM-Podcast: AfD in Sachsen-Anhalt - über Mac...,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [18]:
text_col = 'Headline'
label_cols = df_gold.columns[2:]

print(label_cols)
print("Number of labels:", len(label_cols))

Index(['Economic', 'Capacity and resources', 'Morality',
       'Fairness and equality',
       'Legality, constitutionality and jurisprudence',
       'Policy prescription and evaluation', 'Crime and punishment',
       'Security and defense', 'Health and safety', 'Quality of life',
       'Cultural identity', 'Public opinion', 'Political',
       'External regulation and reputation', 'other'],
      dtype='object')
Number of labels: 15


In [19]:
# Tokenize new headlines
inputs = tokenizer(
    df_predict.tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Move inputs to same device as model
inputs = {key: value.to(device) for key, value in inputs.items()}

# Prediction
with torch.no_grad():
    outputs = model(**inputs)

# Convert logits -> probabilities
probs = torch.sigmoid(outputs.logits)

# Convert probabilities -> 0/1 using the same 0.5 threshold
predictions = (probs >= 0.5).int().cpu().numpy()

In [20]:
df_preds = pd.DataFrame(
    predictions,
    columns=label_cols
)

# Put the original headlines as the first column
df_preds.insert(
    0,
    "Headline",
    df_predict.values
)

df_preds

,Headline,Economic,Capacity and resources,Morality,Fairness and equality,"Legality, constitutionality and jurisprudence",Policy prescription and evaluation,Crime and punishment,Security and defense,Health and safety,Quality of life,Cultural identity,Public opinion,Political,External regulation and reputation,other
0,Warum die SPD jetzt auf Manuela Schwesig hofft,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,Livestream: Die Nachrichten auf tagesschau24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,Hohe Spritpreise: So will die Bundesregierung ...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Jemen: Warum die Huthi-Miliz den Welthandel ge...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
4,Kosovos Ex-Präsident Thaçi zu 25 Jahren Haft v...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,Marktbericht: DAX rückt nach US-Zinsentscheid ...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
66,"Wetter Deutschland: Wolken, Temperaturen, Wind...",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
67,Regenradar Deutschland: Aktueller Niederschlag,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
68,11KM-Podcast: AfD in Sachsen-Anhalt - über Mac...,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [21]:
df_preds.iloc[:, 1:]

,Economic,Capacity and resources,Morality,Fairness and equality,"Legality, constitutionality and jurisprudence",Policy prescription and evaluation,Crime and punishment,Security and defense,Health and safety,Quality of life,Cultural identity,Public opinion,Political,External regulation and reputation,other
0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
66,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
67,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
68,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [22]:
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support

# Sample binary prediction data (15 labels/columns)
# y_true and y_pred should be DataFrames containing 0s and 1s
y_pred = df_preds.iloc[:, 1:]
y_true = df_gold[label_cols]

# 1. Aggregate Micro Metrics (Global across all 15 labels)
micro_prec, micro_rec, micro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="micro"
)

# 2. Aggregate Macro Metrics (Unweighted mean across all 15 labels)
macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro"
)

# 3. Accuracy Metrics
subset_acc = accuracy_score(y_true, y_pred)          # Exact Match Ratio (all 15 must match)
hamming_acc = (y_true == y_pred).to_numpy().mean()   # Element-wise / Label-level accuracy

# 4. Per-Group / Per-Label Metrics (Precision, Recall, F1, and Support)
per_label_report = classification_report(
    y_true, y_pred, target_names=y_true.columns, output_dict=True
)
per_label_df = pd.DataFrame(per_label_report).transpose().loc[
    y_true.columns, ["precision", "recall", "f1-score"]
]

# --- Display Results ---
print("---- AGGREGATE METRICS")
print(f"Micro Precision:  {micro_prec:.4f}")
print(f"Micro Recall:     {micro_rec:.4f}")
print(f"Micro F1:         {micro_f1:.4f}")
print("-------------------------")
print(f"Macro Precision:  {macro_prec:.4f}")
print(f"Macro Recall:     {macro_rec:.4f}")
print(f"Macro F1:         {macro_f1:.4f}")
print("-------------------------")
print(f"Subset Accuracy:  {subset_acc:.4f}  (Exact match across all labels)")
print(f"Hamming Accuracy: {hamming_acc:.4f}  (Element-wise accuracy)")

print(f"\n---- PER-LABEL\n{per_label_df}")

---- AGGREGATE METRICS
Micro Precision:  0.7692
Micro Recall:     0.4762
Micro F1:         0.5882
-------------------------
Macro Precision:  0.4844
Macro Recall:     0.3443
Macro F1:         0.3794
-------------------------
Subset Accuracy:  0.4000  (Exact match across all labels)
Hamming Accuracy: 0.9467  (Element-wise accuracy)

---- PER-LABEL
                                               precision    recall  f1-score
Economic                                        0.833333  0.666667  0.740741
Capacity and resources                          0.500000  0.166667  0.250000
Morality                                        0.000000  0.000000  0.000000
Fairness and equality                           0.000000  0.000000  0.000000
Legality, constitutionality and jurisprudence   1.000000  1.000000  1.000000
Policy prescription and evaluation              0.833333  0.625000  0.714286
Crime and punishment                            0.000000  0.000000  0.000000
Security and defense               

In [24]:
sums = y_pred.sum(axis=1)

result = {
0: (sums == 0).sum(),
1: (sums == 1).sum(),
2: (sums == 2).sum(),
3: (sums == 3).sum(),
'>3': (sums > 3).sum()
}

print(result)

{0: np.int64(23), 1: np.int64(42), 2: np.int64(5), 3: np.int64(0), '>3': np.int64(0)}


# 2 Anwendung: Extraktion deutscher Nachrichten über RSS

## 2.1 Nachrichten Extrahieren

In [1]:
!python --version

Python 3.13.15


In [1]:
%pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.1 MB/s eta 0:00:00


In [2]:
%pip install feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import nltk
nltk.download('stopwords')
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
import feedparser
import numpy as np

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [25]:
german_stop_words = stopwords.words('german')
vectorizer_model = CountVectorizer(stop_words=german_stop_words)

In [26]:
# Tagesschau RSS Feed URL (or replace with your proxy link)
URL_tagesschau = 'https://www.tagesschau.de/index~rss2.xml'
URL_faz = 'https://www.faz.net/rss/aktuell/'
URL_taz = 'https://taz.de/!p4608;rss/'

# Parse the RSS feed
feed_tagesschau = feedparser.parse(URL_tagesschau)
feed_faz = feedparser.parse(URL_faz)
feed_taz = feedparser.parse(URL_taz)

# Extract titles into a list
headlines_tagesschau = list(set([entry.title for entry in feed_tagesschau.entries]))
headlines_faz = list(set([entry.title for entry in feed_faz.entries]))
headlines_taz = list(set([entry.title for entry in feed_taz.entries]))

display(len(headlines_tagesschau))
display(headlines_tagesschau[:5])

display(len(headlines_faz))
display(headlines_faz[:5])

display(len(headlines_taz))
display(headlines_taz[:5])

64

['Klaus J. Behrendt hört beim Kölner Tatort auf',
 'Los Angeles: Lucas Museum of Narrative Art wird eröffnet',
 'Merz in der Unionsfraktion: Ein weiterer ungemütlicher Arbeitstag',
 '11KM-Podcast: VW in der Krise - kann der Autoriese überleben?',
 'Linke in Berlin: Zwischen Regierungsbildung und Antisemitismus-Debatte']

175

['Kolumne „Midlife-Crisis“: So schlecht (wie mit 40) ging es mir noch nie',
 'Russlands Scheinwahl: So soll Putin die Wahl gefälscht haben',
 'Inspiration aus der Antike: Olympische Spiele treiben den Städtebau an',
 'Vermögensaufbau: Wie wir reich wurden',
 'Einigung in Brüssel: EU-Staaten streichen zwei Oligarchen von Sanktionsliste']

50

['Neue Blüte im deutschen Tennis: Alex Zverevs später Sommer',
 'Musiker Stephan Eicher: „Ich finde, dass Haarlänge ein Diskussionsgrund ist“',
 'CSU-Klausur in Kloster Banz: Söder droht Berlin',
 'Comeback der Sitcom mit „Leanne“: Humor in Krisenzeiten',
 '+++ Nachrichten im Ukrainekrieg +++: Tote bei russischem Luftangriff auf Industriestadt Dnipro']

In [27]:
all_headlines = {
    'mitte': headlines_tagesschau,
    'konserv': headlines_faz,
    "progress": headlines_taz
}

In [28]:
items = []
political_orient = []

for orient, lst in all_headlines.items():
    for item in lst:
        items.append(item)
        political_orient.append(orient)

df_all_headlines = pd.DataFrame({
    'title': items,
    'orientation': political_orient
})

df_all_headlines.head()

,title,orientation
0,Klaus J. Behrendt hört beim Kölner Tatort auf,mitte
1,Los Angeles: Lucas Museum of Narrative Art wir...,mitte
2,Merz in der Unionsfraktion: Ein weiterer ungem...,mitte
3,11KM-Podcast: VW in der Krise - kann der Autor...,mitte
4,Linke in Berlin: Zwischen Regierungsbildung un...,mitte


In [29]:
len(df_all_headlines)

289

In [15]:
df_all_headlines.to_excel('all_headlines_260922.xlsx')

## 2.2 Cluster bilden

In [31]:
df_all_headlines = pd.read_excel('all_headlines_260922.xlsx')

In [77]:
len(df_all_headlines)

281

In [32]:
# 1. UMAP: Force high-dimensional preservation of local patterns
umap_model = UMAP(
    n_neighbors=10,       # Lower values (e.g., 5-15) focus on local structure
    n_components=10,       # Dimensions for HDBSCAN input
    min_dist=0.1,         # Keep points tight
    metric='cosine',
    random_state=42
)

# 2. HDBSCAN: Lower minimum cluster size for smaller, tighter groups
hdbscan_model = HDBSCAN(
    min_cluster_size=3,   # Reduce this (e.g., 3-10) to get more fine-grained topics
    min_samples=5,        # Lower values make clustering less conservative (less noise/outliers)
    metric='euclidean',
    cluster_selection_method='eom'  # Or 'leaf' for even smaller micro-clusters
)

# 3. Fit BERTopic with custom sub-models
topic_model = BERTopic(
    embedding_model="paraphrase-multilingual-MiniLM-L12-v2",
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=None         # Ensure auto-reduction is turned off
)

In [33]:
topics, _ = topic_model.fit_transform(df_all_headlines['title'])

# 3. Create a working DataFrame mapping each headline to its assigned topic
df_topics = pd.DataFrame({"headline": df_all_headlines['title'], "topic": topics})

# Filter out noise/unclustered documents (topic -1) if present
df_topics = df_topics[df_topics["topic"] != -1]

# 4. Find the centroid headline for each cluster
# Extract all embeddings generated by the underlying model
embeddings = topic_model._extract_embeddings(df_topics["headline"].tolist())
df_topics["embedding"] = list(embeddings)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [80]:
df_topics

,headline,topic,embedding,centroid_name
0,Klage gegen Paramount: Streit über Warner-Über...,3,"[0.04694377, 0.10251266, -0.24056399, -0.28678...",Actionfilm „Runner“: Kloppen war früher besser
1,SPD nach den Wahlen: Das neue Machtzentrum Sch...,1,"[-0.06191804, -0.017312327, -0.24934268, 0.119...",Nach den wahlen im Osten: Für CDU und SPD gilt...
2,tagesschau in 100 Sekunden,8,"[-0.00803826, 0.09338033, 0.062267214, -0.2905...",tagesschau
3,"15 Minuten-Podcast: Gewalt im ÖPNV, mehr Start...",3,"[-0.01979613, -0.24341825, 0.016768752, -0.077...",Actionfilm „Runner“: Kloppen war früher besser
5,tagesschau in Einfacher Sprache,8,"[-0.09719487, -0.16393444, 0.045483693, -0.114...",tagesschau
...,...,...,...,...
276,Nach Terrorangriff im Westjordanland: Steht Is...,7,"[0.059777018, 0.48952386, -0.18248866, -0.2746...",Islamistische Gefährder in Bremen: Schießen le...
277,+++ Nachrichten im Ukrainekrieg +++: Tote bei ...,5,"[-0.104872875, 0.17471465, -0.28605843, -0.022...",Russland: Kremlpartei liegt bei Duma-Wahl vorn...
278,Chat mit Mitglied des Remmo-Clans: Ferat Koçak...,0,"[-0.025027914, 0.19966178, -0.034878943, 0.133...",Jung­wäh­le­r*in­nen in Berlin: Das Herz schlä...
279,Antifaschistin über Wahlen in MV: „Das Ergebni...,1,"[-0.07395053, 0.25391585, -0.2585911, 0.121645...",Nach den wahlen im Osten: Für CDU und SPD gilt...


In [81]:
sum(pd.Series(topics) == -1)

70

## 2.3 Evaluation der Clusterkohärenz

In [34]:
topic_coherence = {}

for topic_id, group in df_topics.groupby("topic"):

    embeddings = np.vstack(group["embedding"].values)

    # Topic centroid
    centroid = embeddings.mean(axis=0, keepdims=True)

    # Similarity of every document to centroid
    similarities = cosine_similarity(
        embeddings,
        centroid
    ).flatten()

    topic_coherence[topic_id] = {
        "n_documents": len(group),
        "mean_cosine": similarities.mean(),
        "median_cosine": np.median(similarities),
        "min_cosine": similarities.min()
    }

semantic_coherence_df = pd.DataFrame.from_dict(
    topic_coherence,
    orient="index"
)

semantic_coherence_df.index.name = "Topic"

print(semantic_coherence_df)
print(f"Mean coherence: {semantic_coherence_df['mean_cosine'].mean():.3f}")

       n_documents  mean_cosine  median_cosine  min_cosine
Topic                                                     
0               47     0.683212       0.710306    0.432899
1               35     0.637418       0.679585    0.279559
2               17     0.623741       0.654688    0.337038
3               16     0.666077       0.663140    0.466549
4               15     0.618498       0.620668    0.358739
5               11     0.824117       0.846366    0.664970
6               10     0.765653       0.798400    0.526862
7                9     0.764164       0.803490    0.651421
8                9     0.705427       0.719266    0.598214
9                8     0.702610       0.709430    0.553456
10               6     0.800209       0.819654    0.724943
11               6     0.815816       0.830540    0.721596
12               6     0.758077       0.762571    0.687666
13               5     0.777897       0.787905    0.715176
14               4     0.751527       0.754855    0.6809

In [35]:
semantic_coherence_df["mean_cosine"].describe()

,mean_cosine
count,17.000000
mean,0.733252
std,0.067570
min,0.618498
25%,0.683212
50%,0.758077
75%,0.781663
max,0.824117


In [36]:
X = np.vstack(df_topics["embedding"].values)
labels = df_topics["topic"].values

silhouette = silhouette_score(
    X,
    labels,
    metric="cosine"
)

print(f"Cosine silhouette: {silhouette:.4f}")

# The model is finding groups of semantically related headlines, but those groups are not very clearly separated from each other.

Cosine silhouette: 0.1419


In [37]:
centroid_map = {}

for topic_id, group in df_topics.groupby("topic"):
    topic_embeddings = list(group["embedding"])

    centroid_vec = [
        sum(col) / len(col)
        for col in zip(*topic_embeddings)
    ]

    sims = cosine_similarity(
        topic_embeddings,
        [centroid_vec]
    ).flatten()

    representative_idx = sims.argmax()

    representative_headline = group.iloc[representative_idx]["headline"]

    centroid_map[topic_id] = representative_headline

In [38]:
df_topics["centroid_name"] = df_topics["topic"].map(centroid_map)

In [60]:
cluster_dict = {}

for topic_id, group in df_topics.groupby("topic"):

    topic_embeddings = list(group["embedding"])

    centroid_vec = np.mean(topic_embeddings, axis=0)

    sims = cosine_similarity(
        topic_embeddings,
        [centroid_vec]
    ).flatten()

    representative_idx = sims.argmax()
    representative_headline = group.iloc[representative_idx]["headline"]

    all_headlines = group["headline"].tolist()

    column_name = f"Topic {topic_id}: {representative_headline}"

    cluster_dict[column_name] = pd.Series(all_headlines)

final_df = pd.DataFrame(cluster_dict)

In [61]:
final_df

,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das Herz schlägt links,Topic 1: Nach den wahlen im Osten: Für CDU und SPD gilt: Jetzt nicht die Nerven verlieren,Topic 2: Klimaschutz: „So ein heißer Sommer kann ein Lerneffekt sein“,Topic 3: Actionfilm „Runner“: Kloppen war früher besser,Topic 4: Gesunde Ernährung: „Mit der richtigen Ernährung kann man mehrere Lebensjahre rausschlagen“,Topic 5: Russland: Kremlpartei liegt bei Duma-Wahl vorne - wie erwartet,Topic 6: Medien im Weißen Haus: US-Sender nehmen Trump „off air“ in Solidarität mit CNN,Topic 7: Islamistische Gefährder in Bremen: Schießen lernen und auf Insta posten,Topic 8: tagesschau,Topic 9: Kolumne „Midlife Crisis“: Frauen in der Lebensmitte wollen Sex – nur anderen als früher,"Topic 10: Debatte über Krankenkassensystem: ""Merz hat recht""",Topic 11: Teurer Wohnungsbau: „Eine Miete von 20 Euro ist eine magische Grenze“,Topic 12: Vermögensaufbau: Wie wir reich wurden,Topic 13: Gutverdiener in Geldnot: „10.000 Euro im Monat reichen nicht – was tun?“,Topic 14: Managementprofi im Interview: „Ich habe sieben große Effizienzkiller ausgemacht“,"Topic 15: Tote und Vermisste nach Taifun ""Dujuan"" in Japan",Topic 16: Datenschutz-Verstöße: Google muss 400 Millionen Euro Strafe zahlen
0,Regenradar Deutschland: Aktueller Niederschlag,SPD nach den Wahlen: Das neue Machtzentrum Sch...,Chemiekonzern Evonik will 3.200 Stellen abbauen,Klage gegen Paramount: Streit über Warner-Über...,Psychische Probleme: „Mit Burn-out dürfte man ...,Kremlpartei Geeintes Russland gewinnt Duma-Wahl,Selenskyj kündigt Treffen mit Trump diese Woch...,Iran-Krieg und Last-Minute-Trend zehren an TUI...,tagesschau in 100 Sekunden,Lust in der Lebensmitte: „Es gibt eine gute St...,"""Gerechtigkeitsproblem"": Merz tritt Debatte üb...",Immobilien in Dörfern: 95.000 Euro für das Hau...,Wunderportfolio 2.0: Warum das neue ETF-Portfo...,Neuauflage des Tankrabatts: Steuern sollen um ...,Unternehmen sehen Bürokratie als größtes Stand...,Tote und Vermisste nach Taifun in Japan,E-Mails zeigen: Ministerium nannte niedrigeren...
1,Linke in Berlin: Zwischen Regierungsbildung un...,Was die Wahlergebnisse für die Regierungsbildu...,Kabinett bringt Tankrabatt auf den Weg,"15 Minuten-Podcast: Gewalt im ÖPNV, mehr Start...",Mein Weg zum Triathlon (1): „Wenn ich jetzt ni...,Duma-Wahl: Putins Sieg an der Zahlenfront,UN-Generaldebatte in New York - Rede von Trump...,Im Hochsauerland: 15-Jähriger getötet – öffent...,tagesschau in Einfacher Sprache,Wenn die Lust fehlt: „Das Problem ist: Paare g...,"Debatte über Krankenkassensystem: ""Merz hat re...",Studie zu 23 Städten: Wo die Gefahr von Immobi...,Finanzkrisen: Wie Privatmärkte mit Krisen fert...,Gutverdiener in Geldnot: „10.000 Euro im Monat...,Weiterbildung: Die besten KI-Kurse für Einstei...,"Tote und Vermisste nach Taifun ""Dujuan"" in Japan",Datenschutz-Verstöße: Google muss 400 Millione...
2,AfD-Wahlerfolg in Sachsen-Anhalt: Steht die Ku...,"CDU-Wahlschlappe: Neuer Kurs, alte Probleme",Windenergiebranche: Milliardenpoker um Offshor...,Los Angeles: Lucas Museum of Narrative Art wir...,Gesunde Ernährung: „Mit der richtigen Ernährun...,Ficos Balanceakt zwischen Kreml-Nähe und NATO-...,"UN wollen neues ""Rolltreppen-Gate"" mit Trump v...",Kriegsverbrechen: Lauter störende Vergangenheiten,tagesschau,Kolumne „Midlife Crisis“: Frauen in der Lebens...,Gesundheitsökonom: Gesetzliche und private Ver...,Immobilien vor Gericht: Geschäft mit Kunstwerk...,Vermögensaufbau: Reich werden mit den besten T...,Sozialversicherung: Bis zu 2915 Euro höhere Ab...,Managementprofi im Interview: „Ich habe sieben...,Kalifornien: Löschhubschrauber stürzt im Yosem...,Auktion in Köln: Auf Papier gebannt durch Zeit...
3,Wahlparty in Berlin: Linken-Politiker Kocak di...,EU und Philippinen einigen sich auf Freihandel...,Offshore-Projekte von BP drohen zu scheitern,Livestream: Die Nachrichten auf tagesschau24,So läuft Sugardating: „Ich gebe im Monat 1600 ...,Russland: Kremlpartei liegt bei Duma-Wahl vorn...,Trumps Str

In [76]:
final_df.count().sum()

np.int64(211)

## 2.3 Labelvorhersage

In [41]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "nameless-0/GER_frame-detect_mgbert134M"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [42]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [43]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded on:", device)
print("Number of labels:", model.config.num_labels)

Model loaded on: cuda
Number of labels: 15


In [44]:
# Recreate the same label column definition used during training
df_LLM = pd.read_excel("titelannotationen_gesamt_normalisiert.xlsx")

df_LLM15 = df_LLM.copy()

cols = df_LLM.columns[4:]

df_LLM15['other'] = (
    df_LLM15[cols]
    .eq(0)
    .all(axis=1)
    .astype(int)
)

df_LLM15 = df_LLM15[
    (df_LLM15['other'] == 0) |
    df_LLM15['Language'].isin(['ge', 'chn'])
]

df_LLM_comp = df_LLM15.drop(
    columns=['row_id', 'article_id', 'Language']
)

text_col = 'Headline_GER'
label_cols = [col for col in df_LLM_comp.columns if col != text_col]

print(label_cols)
print("Number of labels:", len(label_cols))

['Economic', 'Capacity and resources', 'Morality', 'Fairness and equality', 'Legality, constitutionality and jurisprudence', 'Policy prescription and evaluation', 'Crime and punishment', 'Security and defense', 'Health and safety', 'Quality of life', 'Cultural identity', 'Public opinion', 'Political', 'External regulation and reputation', 'other']
Number of labels: 15


In [45]:
df_new = pd.DataFrame({
    "headline": df_all_headlines['title']
})

df_new

,headline
0,Klage gegen Paramount: Streit über Warner-Über...
1,SPD nach den Wahlen: Das neue Machtzentrum Sch...
2,tagesschau in 100 Sekunden
3,"15 Minuten-Podcast: Gewalt im ÖPNV, mehr Start..."
4,Heizkosten steigen 2026 laut Prognose an
...,...
276,Nach Terrorangriff im Westjordanland: Steht Is...
277,+++ Nachrichten im Ukrainekrieg +++: Tote bei ...
278,Chat mit Mitglied des Remmo-Clans: Ferat Koçak...
279,Antifaschistin über Wahlen in MV: „Das Ergebni...


In [46]:
# Tokenize new headlines
inputs = tokenizer(
    df_new["headline"].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Move inputs to same device as model
inputs = {key: value.to(device) for key, value in inputs.items()}

# Prediction
with torch.no_grad():
    outputs = model(**inputs)

# Convert logits -> probabilities
probs = torch.sigmoid(outputs.logits)

# Convert probabilities -> 0/1 using the same 0.5 threshold
predictions = (probs >= 0.5).int().cpu().numpy()

In [47]:
df_predictions = pd.DataFrame(
    predictions,
    columns=label_cols
)

# Put the original headlines as the first column
df_predictions.insert(
    0,
    "headline",
    df_new["headline"].values
)

df_predictions

,headline,Economic,Capacity and resources,Morality,Fairness and equality,"Legality, constitutionality and jurisprudence",Policy prescription and evaluation,Crime and punishment,Security and defense,Health and safety,Quality of life,Cultural identity,Public opinion,Political,External regulation and reputation,other
0,Klage gegen Paramount: Streit über Warner-Über...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,SPD nach den Wahlen: Das neue Machtzentrum Sch...,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,tagesschau in 100 Sekunden,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,"15 Minuten-Podcast: Gewalt im ÖPNV, mehr Start...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,Heizkosten steigen 2026 laut Prognose an,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,Nach Terrorangriff im Westjordanland: Steht Is...,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0
277,+++ Nachrichten im Ukrainekrieg +++: Tote bei ...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
278,Chat mit Mitglied des Remmo-Clans: Ferat Koçak...,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
279,Antifaschistin über Wahlen in MV: „Das Ergebni...,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [48]:
print(len(df_all_headlines))
print(len(df_predictions))

281
281


## 2.4 0-Class Label bereinigen

In [49]:
import pandas as pd

print('len:',len(df_predictions))
mask = (df_predictions.iloc[:, 1:] == 0).all(axis=1)
count = mask.sum()
print('count:', count)

len: 281
count: 118


In [50]:
label_cols_only = [col for col in df_predictions.columns if col != 'headline']

df_long = df_predictions.copy()

df_long['label'] = df_long[label_cols_only].apply(
    lambda row: '+'.join(row.index[row == 1]),
    axis=1
)

# Filter out headlines that have no labels assigned (where the 'label' column is an empty string)
df_long = df_long[df_long['label'] != '']

df_long = df_long[['headline', 'label']].reset_index(drop=True)

df_long

,headline,label
0,Klage gegen Paramount: Streit über Warner-Über...,"Legality, constitutionality and jurisprudence"
1,SPD nach den Wahlen: Das neue Machtzentrum Sch...,Political
2,"15 Minuten-Podcast: Gewalt im ÖPNV, mehr Start...",other
3,Heizkosten steigen 2026 laut Prognose an,Economic
4,Was die Wahlergebnisse für die Regierungsbildu...,Political
...,...,...
158,Jung­wäh­le­r*in­nen in Berlin: Das Herz schlä...,Political
159,Nach Terrorangriff im Westjordanland: Steht Is...,Crime and punishment+Security and defense
160,+++ Nachrichten im Ukrainekrieg +++: Tote bei ...,Security and defense
161,Chat mit Mitglied des Remmo-Clans: Ferat Koçak...,Political


In [51]:
print(len(df_long))

163


In [ ]:
zero_rows = df_predictions[(df_predictions.iloc[:, 1:] == 0).all(axis=1)]

pd.set_option('display.max_colwidth', None)
zero_rows['headline']

,headline
7,"Tina Hassel, ARD Brüssel, zzt. Straßburg, zum Social-Media-Verbot für unter 13-Jährige"
16,Jürgen Klopp überrascht mit erstem DFB-Kader und nominiert 44 Spieler
17,Kimmich ist zurück auf seiner Lieblingsposition
19,"""Klopp will alle Spieler prüfen"""
20,Darum dauert die Länderspielpause fast drei Wochen
...,...
286,Spielfilm „Pressure“ über D-Day: Druck von allen Seiten
287,15 Jahre Occupy Wall Street: Als die Wall Street abgeriegelt wurde
293,Maßnahmen nach Anschlag auf CSD: Repression und ein bisschen Prävention
302,Berliner Linke fordert Luxusvillensteuer: 3.000 Luxusimmobilien wären in Berlin-Grunewald betroffen


## 2.5 Merge mit politischer Orientierung

In [88]:
merged_df = pd.merge(df_long, df_all_headlines, left_on='headline', right_on='title').drop(columns=['title', 'Unnamed: 0'])
merged_df

,headline,label,orientation
0,Klage gegen Paramount: Streit über Warner-Über...,"Legality, constitutionality and jurisprudence",mitte
1,SPD nach den Wahlen: Das neue Machtzentrum Sch...,Political,mitte
2,"15 Minuten-Podcast: Gewalt im ÖPNV, mehr Start...",other,mitte
3,Heizkosten steigen 2026 laut Prognose an,Economic,mitte
4,Was die Wahlergebnisse für die Regierungsbildu...,Political,mitte
...,...,...,...
158,Jung­wäh­le­r*in­nen in Berlin: Das Herz schlä...,Political,progress
159,Nach Terrorangriff im Westjordanland: Steht Is...,Crime and punishment+Security and defense,progress
160,+++ Nachrichten im Ukrainekrieg +++: Tote bei ...,Security and defense,progress
161,Chat mit Mitglied des Remmo-Clans: Ferat Koçak...,Political,progress


## 2.6 Merge mit Cluster

In [62]:
# 1. Reshape cluster table from wide to long (tidy) format
long_cluster = final_df.melt(
    var_name='cluster_headline',
    value_name='headline'
).dropna(subset=['headline'])

long_cluster

,cluster_headline,headline
0,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,Regenradar Deutschland: Aktueller Niederschlag
1,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,Linke in Berlin: Zwischen Regierungsbildung un...
2,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,AfD-Wahlerfolg in Sachsen-Anhalt: Steht die Ku...
3,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,Wahlparty in Berlin: Linken-Politiker Kocak di...
4,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,Reaktionen auf Wahlen: Deutschland sorgt gerad...
...,...,...
707,"Topic 15: Tote und Vermisste nach Taifun ""Duju...",Kalifornien: Löschhubschrauber stürzt im Yosem...
708,"Topic 15: Tote und Vermisste nach Taifun ""Duju...",Triebwerksausfall: Segelflug mit einem Airbus
752,Topic 16: Datenschutz-Verstöße: Google muss 40...,E-Mails zeigen: Ministerium nannte niedrigeren...
753,Topic 16: Datenschutz-Verstöße: Google muss 40...,Datenschutz-Verstöße: Google muss 400 Millione...


In [90]:
# 2. Merge metadata for the individual headlines (Intersection filter)
merged_cluster = pd.merge(
    long_cluster,
    merged_df,
    on='headline',
    how='inner'
)

news_overview = merged_cluster.drop_duplicates()

In [91]:
news_overview.head()

,cluster_headline,headline,label,orientation
0,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,Linke in Berlin: Zwischen Regierungsbildung un...,Political,mitte
1,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,AfD-Wahlerfolg in Sachsen-Anhalt: Steht die Ku...,Political,mitte
2,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,Wahlparty in Berlin: Linken-Politiker Kocak di...,Political,mitte
3,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,Reaktionen auf Wahlen: Deutschland sorgt gerad...,Political,mitte
4,Topic 0: Jung­wäh­le­r*in­nen in Berlin: Das H...,"Wetter Deutschland: Wolken, Temperaturen, Wind...",Capacity and resources,mitte


In [84]:
len(news_overview)

131

In [92]:
news_overview.to_excel('news_overview.xlsx')